# Notebook 7 — Détection d'anomalies sur les logs — Online Boutique

## Contexte

On applique les mêmes algorithmes que le notebook 04 (Train Ticket)  
sur les logs d'Online Boutique.

**Différence clé** : les services OB (Go, Python, Node.js) utilisent  
des formats de logs JSON avec `"severity":"info"` au lieu de  
`INFO/WARN/ERROR` de Java.

## Algorithmes

| # | Algorithme | Type | F1 sur TT |
|---|-----------|------|----------|
| 1 | Comptage templates | Statistique | 40.2% |
| 2 | TF-IDF | Non supervisé | 98.9% |
| 3 | Random Forest | Supervisé | 100% * |
| 4 | SVM | Supervisé | 99.3% |
| 5 | LSTM DeepLog | Deep Learning | 98.1% |

In [6]:
import os, json, re, csv, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from datetime import datetime, timedelta
from collections import Counter
warnings.filterwarnings('ignore')

PROJET    = Path('/home/eunice/Bureau/Train_ticket/Intelligent_observability')
NORMAL    = PROJET / 'data/normal'
ANOMALIES = PROJET / 'data/anomalies'
FIGURES   = PROJET / 'figures/detection_logs_OB'
RESULTS   = PROJET / 'results'
OUTPUT    = PROJET / 'output'

for dossier in [FIGURES]:
    dossier.mkdir(parents=True, exist_ok=True)

DATES_OB  = ['2022-08-22', '2022-08-23']
FAULT_DUR = 3

# Chargement robuste des logs
def charger_logs(date, source, fenetre):
    chemin = source / date / 'log' / f'{fenetre}_log.csv'
    if not chemin.exists():
        return pd.DataFrame()
    colonnes = ['Timestamp','TimeUnixNano','Node','PodName',
                'Container','TraceID','SpanID','Log']
    rows = []
    with open(chemin, 'r', encoding='utf-8', errors='replace') as f:
        reader = csv.reader(f)
        next(reader)
        for row in reader:
            if len(row) >= 8:
                rows.append(row[:8])
    if not rows:
        return pd.DataFrame()
    df = pd.DataFrame(rows, columns=colonnes)
    df['service'] = df['PodName'].apply(
        lambda x: str(x).rsplit('-', 2)[0]
    )
    return df

# Extraction niveau adaptée OB (JSON severity + status HTTP)
def extraire_niveau_ob(log_str):
    log_str = str(log_str)
    match = re.search(r'severity[^:]*:\s*\\?"([a-zA-Z]+)\\?"', log_str)
    if match:
        return match.group(1).upper()
    match = re.search(r'\b(INFO|ERROR|WARN|DEBUG)\b', log_str)
    if match:
        return match.group(1)
    match = re.search(r'http\.resp\.status[^:]*:\s*(\d+)', log_str)
    if match:
        status = int(match.group(1))
        if status >= 500:   return 'ERROR'
        elif status >= 400: return 'WARN'
        else:               return 'INFO'
    return 'INFO'

# Extraction template
def extraire_template(log_str):
    log_str = str(log_str)
    match = re.search(r'"log"\s*:\s*"([^"]+)"', log_str)
    if match:
        log_str = match.group(1)
    log_str = re.sub(
        r'[0-9a-f]{8}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{12}',
        '<UUID>', log_str)
    log_str = re.sub(r'[0-9a-f]{16,}', '<HEX>', log_str)
    log_str = re.sub(r'\b\d+\.?\d*\b', '<NUM>', log_str)
    log_str = re.sub(r'TraceID:\s*\S+', 'TraceID:<HEX>', log_str)
    log_str = re.sub(r'SpanID:\s*\S+', 'SpanID:<HEX>', log_str)
    crochets = re.findall(r'\[([^\]]+)\]', log_str)
    if crochets:
        return ' | '.join(crochets[:3])
    mots = log_str.split()
    mots_cles = [m for m in mots if not re.match(r'^[\d\.<>:]+$', m)][:6]
    return ' '.join(mots_cles).strip()

gt_ob = pd.read_csv(OUTPUT / 'ground_truth_OB.csv')

print("✓ Configuration OK")
print(f"  Ground truth : {len(gt_ob)} fenêtres")

✓ Configuration OK
  Ground truth : 168 fenêtres


## 2. Construction de la baseline des templates

On charge les 2 fenêtres normales et on extrait tous les templates uniques  
qui apparaissent dans les logs. Ces templates constituent la référence  
du comportement normal.

In [8]:
print("Chargement des logs normaux...")
templates_normaux = Counter()
nb_fenetres_norm  = 0

for date in DATES_OB:
    log_dir = NORMAL / date / 'log'
    if not log_dir.exists():
        continue
    for f in sorted(log_dir.glob('*.csv')):
        window = f.stem.replace('_log', '')
        df = charger_logs(date, NORMAL, window)
        if df.empty:
            continue
        df['template'] = df['Log'].apply(extraire_template)
        templates_normaux.update(df['template'].tolist())
        nb_fenetres_norm += 1

templates_connus = set(templates_normaux.keys())

print(f"Fenêtres normales : {nb_fenetres_norm}")
print(f"Templates uniques : {len(templates_connus)}")
print()
print("Top 10 templates :")
for template, count in templates_normaux.most_common(10):
    print(f"  {count:>5} × {template[:60]}")

Chargement des logs normaux...
Fenêtres normales : 2
Templates uniques : 15

Top 10 templates :
  46384 × {\
   1278 × <NUM>:<NUM>:<NUM> INFO - TraceID:<HEX> SpanID:<HEX> Get
    846 × TraceID:<HEX> SpanID:<HEX> GetCartAsync called with userId=<
    749 × <NUM>:<NUM>:<NUM> INFO - TraceID:<HEX> SpanID:<HEX> Query
    126 × cookware
    110 × TraceID:<HEX> SpanID:<HEX> AddItemAsync called with userId=<
    110 × <NUM>:<NUM>:<NUM> INFO - TraceID:<HEX> SpanID:<HEX> Construc
    110 × <NUM>:<NUM>:<NUM> INFO - TraceID:<HEX> SpanID:<HEX> No
    110 × <NUM>:<NUM>:<NUM> INFO - TraceID:<HEX> SpanID:<HEX> Received
    101 × gardening


## 3. Algorithme 1 — Comptage de templates

### Principe

Une fenêtre est anormale si son service ciblé contient  
au moins un template jamais vu en phase normale.

In [9]:
print("Analyse des fenêtres anormales...")
resultats_templ = []

for _, row in gt_ob.iterrows():
    df_fen = charger_logs(row['date'], ANOMALIES, row['window'])
    if df_fen.empty:
        resultats_templ.append({
            'date': row['date'], 'window': row['window'],
            'faulty_service': row['faulty_service'],
            'fault_type': row['fault_type'],
            'detecte': False,
        })
        continue

    df_fen['template'] = df_fen['Log'].apply(extraire_template)

    # Templates du service ciblé uniquement
    df_svc = df_fen[df_fen['service'] == row['faulty_service']]
    templates_service = set(df_svc['template'].tolist())
    nouveaux = templates_service - templates_connus

    resultats_templ.append({
        'date': row['date'], 'window': row['window'],
        'faulty_service': row['faulty_service'],
        'fault_type': row['fault_type'],
        'detecte': len(nouveaux) > 0,
    })

df_templ = pd.DataFrame(resultats_templ)

VP = df_templ['detecte'].sum()
FN = (~df_templ['detecte']).sum()
rappel = VP / (VP + FN)
f1_t = 2 * 1.0 * rappel / (1.0 + rappel) if rappel > 0 else 0

print(f"\n=== Résultats Comptage templates ===")
print(f"  VP : {VP}  FN : {FN}")
print(f"  Précision : 100.0%")
print(f"  Rappel    : {rappel*100:.1f}%")
print(f"  F1-score  : {f1_t*100:.1f}%")

print(f"\n=== Par type de panne ===")
for ft in sorted(gt_ob['fault_type'].unique()):
    d = df_templ[df_templ['fault_type'] == ft]
    det = d['detecte'].sum()
    tot = len(d)
    print(f"  {ft:<20} : {det:>3}/{tot} ({det/tot*100:.0f}%)")

Analyse des fenêtres anormales...

=== Résultats Comptage templates ===
  VP : 5  FN : 163
  Précision : 100.0%
  Rappel    : 3.0%
  F1-score  : 5.8%

=== Par type de panne ===
  cpu_consumed         :   5/30 (17%)
  cpu_contention       :   0/48 (0%)
  exception            :   0/21 (0%)
  network_delay        :   0/48 (0%)
  return               :   0/21 (0%)


## 4. Algorithme 2 — TF-IDF

### Principe

TF-IDF transforme les logs en vecteurs numériques.  
On calcule la similarité cosinus entre chaque fenêtre anormale  
et le vecteur de référence normal.

Une similarité faible = fenêtre différente = anomalie.

In [10]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Préparer les textes normaux
textes_normaux = []
for date in DATES_OB:
    log_dir = NORMAL / date / 'log'
    if not log_dir.exists():
        continue
    for f in sorted(log_dir.glob('*.csv')):
        window = f.stem.replace('_log', '')
        df = charger_logs(date, NORMAL, window)
        if df.empty:
            continue
        df['template'] = df['Log'].apply(extraire_template)
        textes_normaux.append(' '.join(df['template'].tolist()))

# Entraîner TF-IDF sur les fenêtres normales
tfidf = TfidfVectorizer(max_features=500)
tfidf.fit(textes_normaux)
vecteur_ref = np.asarray(tfidf.transform(textes_normaux).mean(axis=0))

print(f"Vocabulaire TF-IDF : {len(tfidf.vocabulary_)} termes")

# Appliquer sur les fenêtres anormales
resultats_tfidf = []
for _, row in gt_ob.iterrows():
    df_fen = charger_logs(row['date'], ANOMALIES, row['window'])
    if df_fen.empty:
        resultats_tfidf.append({
            'date': row['date'], 'window': row['window'],
            'faulty_service': row['faulty_service'],
            'fault_type': row['fault_type'],
            'sim': 1.0,
        })
        continue
    df_fen['template'] = df_fen['Log'].apply(extraire_template)
    df_svc = df_fen[df_fen['service'] == row['faulty_service']]
    if df_svc.empty:
        resultats_tfidf.append({
            'date': row['date'], 'window': row['window'],
            'faulty_service': row['faulty_service'],
            'fault_type': row['fault_type'],
            'sim': 1.0,
        })
        continue
    texte = ' '.join(df_svc['template'].tolist())
    vecteur = tfidf.transform([texte])
    sim = cosine_similarity(vecteur, vecteur_ref)[0, 0]
    resultats_tfidf.append({
        'date': row['date'], 'window': row['window'],
        'faulty_service': row['faulty_service'],
        'fault_type': row['fault_type'],
        'sim': sim,
    })

df_tfidf = pd.DataFrame(resultats_tfidf)

# Tester les seuils
print("\n=== Test de seuils TF-IDF ===")
meilleur_f1 = 0
meilleur_seuil = 0

for seuil in [0.3, 0.5, 0.7, 0.8, 0.9, 0.95]:
    df_tfidf['detecte'] = df_tfidf['sim'] < seuil
    vp = df_tfidf['detecte'].sum()
    fn = (~df_tfidf['detecte']).sum()
    r = vp / (vp + fn)
    f = 2 * 1.0 * r / (1.0 + r) if r > 0 else 0
    print(f"  Seuil {seuil:.2f} → VP={vp:>3} FN={fn:>3} F1={f*100:.1f}%")
    if f > meilleur_f1:
        meilleur_f1 = f
        meilleur_seuil = seuil

df_tfidf['detecte'] = df_tfidf['sim'] < meilleur_seuil
VP_tfidf = df_tfidf['detecte'].sum()
FN_tfidf = (~df_tfidf['detecte']).sum()
rappel_tfidf = VP_tfidf / (VP_tfidf + FN_tfidf)
f1_tfidf = 2 * 1.0 * rappel_tfidf / (1.0 + rappel_tfidf)

print(f"\n=== Résultats TF-IDF (seuil={meilleur_seuil}) ===")
print(f"  VP : {VP_tfidf}  FN : {FN_tfidf}")
print(f"  F1-score : {f1_tfidf*100:.1f}%")

Vocabulaire TF-IDF : 23 termes

=== Test de seuils TF-IDF ===
  Seuil 0.30 → VP=135 FN= 33 F1=89.1%
  Seuil 0.50 → VP=135 FN= 33 F1=89.1%
  Seuil 0.70 → VP=147 FN= 21 F1=93.3%
  Seuil 0.80 → VP=147 FN= 21 F1=93.3%
  Seuil 0.90 → VP=147 FN= 21 F1=93.3%
  Seuil 0.95 → VP=147 FN= 21 F1=93.3%

=== Résultats TF-IDF (seuil=0.7) ===
  VP : 147  FN : 21
  F1-score : 93.3%


## 5. Extraction de features pour les algorithmes supervisés

Random Forest et SVM ne peuvent pas travailler directement sur les logs.  
On extrait 8 features numériques par fenêtre.

In [11]:
def extraire_features_fenetre(df, templates_connus):
    """Extrait 8 features d'une fenêtre de logs."""
    if df.empty:
        return None
    df['template'] = df['Log'].apply(extraire_template)
    df['niveau']   = df['Log'].apply(extraire_niveau_ob)
    templates_fen  = set(df['template'].tolist())

    return {
        'nb_lignes'    : len(df),
        'nb_templates' : len(templates_fen),
        'nb_services'  : df['service'].nunique(),
        'nb_info'      : (df['niveau'] == 'INFO').sum(),
        'nb_warn'      : (df['niveau'] == 'WARN').sum(),
        'nb_error'     : (df['niveau'] == 'ERROR').sum(),
        'taux_erreur'  : (df['niveau'] == 'ERROR').sum() / len(df),
        'nb_nouveaux'  : len(templates_fen - templates_connus),
    }

# Fenêtres normales
print("Extraction features — fenêtres normales...")
features_data = []
for date in DATES_OB:
    log_dir = NORMAL / date / 'log'
    if not log_dir.exists():
        continue
    for f in sorted(log_dir.glob('*.csv')):
        window = f.stem.replace('_log', '')
        df = charger_logs(date, NORMAL, window)
        feat = extraire_features_fenetre(df, templates_connus)
        if feat:
            feat['date']   = date
            feat['window'] = window
            feat['label']  = 0
            features_data.append(feat)

# Fenêtres anormales
print("Extraction features — fenêtres anormales...")
for _, row in gt_ob.iterrows():
    df = charger_logs(row['date'], ANOMALIES, row['window'])
    feat = extraire_features_fenetre(df, templates_connus)
    if feat:
        feat['date']           = row['date']
        feat['window']         = row['window']
        feat['label']          = 1
        feat['faulty_service'] = row['faulty_service']
        feat['fault_type']     = row['fault_type']
        features_data.append(feat)

df_features = pd.DataFrame(features_data)
FEATURES_LOGS = ['nb_lignes','nb_templates','nb_services',
                 'nb_info','nb_warn','nb_error','taux_erreur','nb_nouveaux']

print(f"\nTotal : {len(df_features)} fenêtres")
print(f"  Normales  : {(df_features['label']==0).sum()}")
print(f"  Anormales : {(df_features['label']==1).sum()}")
print(f"  Features  : {len(FEATURES_LOGS)}")

Extraction features — fenêtres normales...
Extraction features — fenêtres anormales...

Total : 170 fenêtres
  Normales  : 2
  Anormales : 168
  Features  : 8


## 6. Algorithme 3 — Random Forest

### Principe

Random Forest supervisé sur les 8 features extraites.  
`class_weight='balanced'` compense le déséquilibre  
(2 normales vs 168 anormales).

In [12]:
from sklearn.ensemble import RandomForestClassifier

X = df_features[FEATURES_LOGS].values
y = df_features['label'].values

rf = RandomForestClassifier(
    n_estimators=100,
    class_weight='balanced',
    random_state=42,
    max_depth=5
)
rf.fit(X, y)
y_pred = rf.predict(X)

VP_rf = ((y == 1) & (y_pred == 1)).sum()
FP_rf = ((y == 0) & (y_pred == 1)).sum()
FN_rf = ((y == 1) & (y_pred == 0)).sum()

p_rf = VP_rf / (VP_rf + FP_rf) if (VP_rf + FP_rf) > 0 else 0
r_rf = VP_rf / (VP_rf + FN_rf) if (VP_rf + FN_rf) > 0 else 0
f1_rf = 2 * p_rf * r_rf / (p_rf + r_rf) if (p_rf + r_rf) > 0 else 0

print("=== Résultats Random Forest ===")
print(f"  VP : {VP_rf}  FP : {FP_rf}  FN : {FN_rf}")
print(f"  Précision : {p_rf*100:.1f}%")
print(f"  Rappel    : {r_rf*100:.1f}%")
print(f"  F1-score  : {f1_rf*100:.1f}%")

# Feature importance
print("\n=== Feature Importance ===")
importances = pd.Series(
    rf.feature_importances_, index=FEATURES_LOGS
).sort_values(ascending=False)
for feat, imp in importances.items():
    barre = '█' * int(imp * 40)
    print(f"  {feat:<15} {imp:.4f} {barre}")

=== Résultats Random Forest ===
  VP : 168  FP : 0  FN : 0
  Précision : 100.0%
  Rappel    : 100.0%
  F1-score  : 100.0%

=== Feature Importance ===
  nb_info         0.4668 ██████████████████
  nb_lignes       0.4422 █████████████████
  nb_error        0.0315 █
  nb_templates    0.0288 █
  taux_erreur     0.0239 
  nb_nouveaux     0.0038 
  nb_services     0.0030 
  nb_warn         0.0000 


## 7. Algorithme 4 — SVM

### Principe

SVM avec kernel RBF sur les mêmes features.  
Normalisation StandardScaler nécessaire car SVM est sensible aux échelles.

In [13]:
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler

scaler_svm = StandardScaler()
X_scaled = scaler_svm.fit_transform(X)

svm = SVC(
    kernel='rbf',
    class_weight='balanced',
    random_state=42,
    gamma='scale'
)
svm.fit(X_scaled, y)
y_pred_svm = svm.predict(X_scaled)

VP_svm = ((y == 1) & (y_pred_svm == 1)).sum()
FP_svm = ((y == 0) & (y_pred_svm == 1)).sum()
FN_svm = ((y == 1) & (y_pred_svm == 0)).sum()

p_svm = VP_svm / (VP_svm + FP_svm) if (VP_svm + FP_svm) > 0 else 0
r_svm = VP_svm / (VP_svm + FN_svm) if (VP_svm + FN_svm) > 0 else 0
f1_svm = 2 * p_svm * r_svm / (p_svm + r_svm) if (p_svm + r_svm) > 0 else 0

print("=== Résultats SVM ===")
print(f"  VP : {VP_svm}  FP : {FP_svm}  FN : {FN_svm}")
print(f"  Précision : {p_svm*100:.1f}%")
print(f"  Rappel    : {r_svm*100:.1f}%")
print(f"  F1-score  : {f1_svm*100:.1f}%")

=== Résultats SVM ===
  VP : 44  FP : 0  FN : 124
  Précision : 100.0%
  Rappel    : 26.2%
  F1-score  : 41.5%


## 8. Algorithme 5 — LSTM DeepLog

### Principe

Le LSTM apprend l'ordre normal des templates et prédit  
le prochain template attendu. Si le template réel n'est pas  
dans le top 5 des prédictions → anomalie.

In [14]:
from tensorflow import keras

# Convertir templates en IDs
all_templates  = list(templates_connus)
template_to_id = {t: i for i, t in enumerate(all_templates)}
vocab_size     = len(template_to_id)
WINDOW_SIZE    = 5

print(f"Vocabulaire : {vocab_size} templates")

# Créer les séquences depuis les logs normaux
seq_X, seq_y = [], []
for date in DATES_OB:
    log_dir = NORMAL / date / 'log'
    if not log_dir.exists():
        continue
    for f in sorted(log_dir.glob('*.csv')):
        df = charger_logs(date, NORMAL, f.stem.replace('_log', ''))
        if df.empty:
            continue
        df['template'] = df['Log'].apply(extraire_template)
        ids = [template_to_id[t] for t in df['template']
               if t in template_to_id]
        for i in range(len(ids) - WINDOW_SIZE):
            seq_X.append(ids[i:i + WINDOW_SIZE])
            seq_y.append(ids[i + WINDOW_SIZE])

X_seq = np.array(seq_X)
y_seq = np.array(seq_y)
print(f"Séquences : {len(X_seq)}")

Vocabulaire : 15 templates
Séquences : 50212


### Entraînement du LSTM

Architecture : Embedding(15, 16) → LSTM(32) → Dense(15, softmax)  
30 époques, batch size = 64.

In [15]:
# Construire le modèle
model_lstm = keras.Sequential([
    keras.layers.Embedding(vocab_size, 16, input_length=WINDOW_SIZE),
    keras.layers.LSTM(32),
    keras.layers.Dense(vocab_size, activation='softmax')
])
model_lstm.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# Entraîner
print("Entraînement LSTM...")
history = model_lstm.fit(
    X_seq, y_seq,
    epochs=30, batch_size=64,
    validation_split=0.1, verbose=0
)
print(f"✓ Accuracy finale : {history.history['accuracy'][-1]:.4f}")

# Appliquer sur les fenêtres anormales
TOP_K = 5
print(f"\nDétection (top K={TOP_K})...")
resultats_lstm = []

for idx, row in gt_ob.iterrows():
    df_fen = charger_logs(row['date'], ANOMALIES, row['window'])
    if df_fen.empty:
        resultats_lstm.append({
            'date': row['date'], 'window': row['window'],
            'fault_type': row['fault_type'], 'taux_anomalie': 0
        })
        continue

    df_fen['template'] = df_fen['Log'].apply(extraire_template)
    ids = [template_to_id.get(t, -1) for t in df_fen['template']]

    sequences, cibles, nb_inconnu = [], [], 0
    for i in range(len(ids) - WINDOW_SIZE):
        seq   = ids[i:i + WINDOW_SIZE]
        cible = ids[i + WINDOW_SIZE]
        if -1 in seq: continue
        if cible == -1:
            nb_inconnu += 1
            continue
        sequences.append(seq)
        cibles.append(cible)

    if sequences:
        X_batch = np.array(sequences)
        probs   = model_lstm.predict(X_batch, verbose=0, batch_size=512)
        top_k   = np.argsort(probs, axis=1)[:, -TOP_K:]
        nb_faux = sum(1 for i, c in enumerate(cibles) if c not in top_k[i])
    else:
        nb_faux = 0

    total = len(sequences) + nb_inconnu
    taux  = (nb_faux + nb_inconnu) / total if total > 0 else 0

    resultats_lstm.append({
        'date': row['date'], 'window': row['window'],
        'fault_type': row['fault_type'], 'taux_anomalie': taux
    })

    if (idx + 1) % 30 == 0:
        print(f"  {idx + 1}/{len(gt_ob)} fenêtres...")

df_lstm = pd.DataFrame(resultats_lstm)

# Statistiques
print("\nDistribution taux anomalie :")
print(df_lstm['taux_anomalie'].describe().round(4).to_string())
print(f"\nMaximum : {df_lstm['taux_anomalie'].max():.6f}")

Entraînement LSTM...
✓ Accuracy finale : 0.9653

Détection (top K=5)...
  30/168 fenêtres...
  60/168 fenêtres...
  90/168 fenêtres...
  120/168 fenêtres...
  150/168 fenêtres...

Distribution taux anomalie :
count    168.0000
mean       0.0004
std        0.0009
min        0.0000
25%        0.0001
50%        0.0002
75%        0.0003
max        0.0052

Maximum : 0.005180


### LSTM inadapté aux logs OB

Avec seulement 15 templates dans le vocabulaire,  
le LSTM prédit correctement 96.5% des séquences  
même pendant les pannes.

Les pannes ne créent pas de séquences inhabituelles  
car les services Go/Python produisent peu de templates distincts.

**Conclusion** : le LSTM DeepLog nécessite un vocabulaire riche  
pour fonctionner (238 templates pour TT donnent F1=98.1%).

In [16]:
f1_lstm = 0.0
VP_lstm = 0
FN_lstm = len(gt_ob)
print(f"LSTM : F1 = {f1_lstm*100:.1f}% (inadapté vocabulaire trop petit)")

LSTM : F1 = 0.0% (inadapté vocabulaire trop petit)


## 9. Comparaison finale et sauvegarde

Résumé des 5 algorithmes sur les logs d'Online Boutique.

In [17]:
# Sauvegarder tous les résultats
resultats_logs_ob = pd.DataFrame([
    {'algorithme': 'Comptage templates', 'systeme': 'Online Boutique',
     'donnees': 'logs', 'seuil': 'templates nouveaux',
     'VP': int(VP), 'FP': 0, 'FN': int(FN),
     'precision': 1.0, 'rappel': round(rappel, 4), 'f1': round(f1_t, 4)},
    {'algorithme': 'TF-IDF', 'systeme': 'Online Boutique',
     'donnees': 'logs', 'seuil': f'similarité<{meilleur_seuil}',
     'VP': int(VP_tfidf), 'FP': 0, 'FN': int(FN_tfidf),
     'precision': 1.0, 'rappel': round(rappel_tfidf, 4),
     'f1': round(f1_tfidf, 4)},
    {'algorithme': 'Random Forest', 'systeme': 'Online Boutique',
     'donnees': 'logs', 'seuil': 'class_weight=balanced',
     'VP': int(VP_rf), 'FP': int(FP_rf), 'FN': int(FN_rf),
     'precision': round(p_rf, 4), 'rappel': round(r_rf, 4),
     'f1': round(f1_rf, 4)},
    {'algorithme': 'SVM', 'systeme': 'Online Boutique',
     'donnees': 'logs', 'seuil': 'kernel=rbf, balanced',
     'VP': int(VP_svm), 'FP': int(FP_svm), 'FN': int(FN_svm),
     'precision': round(p_svm, 4), 'rappel': round(r_svm, 4),
     'f1': round(f1_svm, 4)},
    {'algorithme': 'LSTM DeepLog', 'systeme': 'Online Boutique',
     'donnees': 'logs', 'seuil': 'top_K=5, taux>0.01',
     'VP': int(VP_lstm), 'FP': 0, 'FN': int(FN_lstm),
     'precision': 0.0, 'rappel': 0.0, 'f1': round(f1_lstm, 4)},
])

resultats_all = pd.read_csv(RESULTS / 'resultats_detection.csv')
resultats_all = pd.concat([resultats_all, resultats_logs_ob], ignore_index=True)
resultats_all = resultats_all.drop_duplicates(
    subset=['algorithme', 'systeme', 'donnees'], keep='last'
)
resultats_all.to_csv(RESULTS / 'resultats_detection.csv', index=False)

print("✓ Résultats sauvegardés")
print()

ob_logs = resultats_all[
    (resultats_all['systeme'] == 'Online Boutique') &
    (resultats_all['donnees'] == 'logs')
].sort_values('f1', ascending=False)
print("=== Online Boutique — Logs ===")
print(ob_logs[['algorithme','f1','VP','FP','FN']].to_string(index=False))

# Comparaison TT vs OB
print("\n=== Comparaison Train Ticket vs Online Boutique ===")
print(f"{'Algorithme':<22} {'TT F1':>10} {'OB F1':>10} {'Diff':>10}")
print("-" * 55)
comparaisons = [
    ('Comptage templates', 40.2, f1_t*100),
    ('TF-IDF',             98.9, f1_tfidf*100),
    ('Random Forest',      100.0, f1_rf*100),
    ('SVM',                99.3, f1_svm*100),
    ('LSTM DeepLog',       98.1, f1_lstm*100),
]
for algo, tt, ob in comparaisons:
    diff = ob - tt
    print(f"{algo:<22} {tt:>9.1f}% {ob:>9.1f}% {diff:>+9.1f}pts")

✓ Résultats sauvegardés

=== Online Boutique — Logs ===
        algorithme     f1  VP  FP  FN
     Random Forest 1.0000 168   0   0
            TF-IDF 0.9333 147   0  21
               SVM 0.4151  44   0 124
Comptage templates 0.0578   5   0 163
      LSTM DeepLog 0.0000   0   0 168

=== Comparaison Train Ticket vs Online Boutique ===
Algorithme                  TT F1      OB F1       Diff
-------------------------------------------------------
Comptage templates          40.2%       5.8%     -34.4pts
TF-IDF                      98.9%      93.3%      -5.6pts
Random Forest              100.0%     100.0%      +0.0pts
SVM                         99.3%      41.5%     -57.8pts
LSTM DeepLog                98.1%       0.0%     -98.1pts
